# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to explore, load, and perform initial processing on the [FAIR²](https://sen.science/doi/10.71728/senscience.y7m0-f273) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset metadata and structure are defined by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Install mlcroissant library if not already installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and explore the dataset using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant JSON-LD schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"
# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Display metadata summary
meta = dataset.metadata
print(f"Dataset Title: {meta.name}")
print(f"Description: {meta.description}\n")
print(f"Identifier: {meta.identifier}")
print(f"Version: {meta.version}")
print(f"License: {meta.license}")

## 2. Data Overview
List all available record sets and their fields by `@id`.

*Note: It is a common Croissant convention to reference entities using their `@id`. This ensures that field, record set, and column names remain unambiguous and match the schema.*

In [ ]:
# List all available record sets and their fields, referencing them by @id.
print("Available Record Sets:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- record_set @id: {rs['@id']}")
    if 'field' in rs:
        print("  Fields:")
        for f in rs['field']:
            if isinstance(f, dict):
                print(f"    - field @id: {f.get('@id', '?')} ({f.get('name', '?')})")
            else:
                print(f"    - field @id: {f}")
    else:
        print("  No fields defined.")
    print()

# Store list of record set @id for later code sections
record_set_ids = [rs['@id'] for rs in record_sets]

## 3. Data Extraction
Load table records from specific record sets into pandas DataFrames.

Refer to entities via `@id` as printed above.

In [ ]:
# Extract data from each record set and load into a DataFrame
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for record set: {record_set_id}")
        else:
            print(f"No records found for record set: {record_set_id}")
    except Exception as e:
        print(f"Error loading records for record set {record_set_id}: {e}")

# Display the columns of the first non-empty DataFrame
for record_set_id, df in dataframes.items():
    print(f"\nFirst record set with data: {record_set_id}")
    print("Columns by @id:")
    print(df.columns.tolist())
    display(df.head())
    break

## 4. Exploratory Data Analysis (EDA)
We now demonstrate filtering, normalization, and grouping, referencing fields by their `@id`.

> **Note:** Adjust the `numeric_field_id` and `group_field_id` with valid `@id`s from the displayed DataFrame columns above for meaningful results.

In [ ]:
# Example parameters -- set these to match fields in the chosen record set
# Replace with actual @id values as printed above!
record_set_id = next(iter(dataframes.keys())) if dataframes else None
if record_set_id is not None:
    df = dataframes[record_set_id]

    # List numeric-looking columns for possible operations
    print("Available columns for numeric analysis:")
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            print(f"- {col}")

    # Select a numeric field by its @id (Update as appropriate):
    numeric_field_id = df.select_dtypes(include=['number']).columns[0] if len(df.select_dtypes(include=['number']).columns) > 0 else None
    if numeric_field_id:
        # Filter records above a threshold
        threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"\nFiltered records where '{numeric_field_id}' > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the chosen numeric field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}':")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Group by another field (categorical/group field)
        # Attempt to choose first non-numeric column
        group_field_id = df.select_dtypes(exclude=['number']).columns[0] if len(df.select_dtypes(exclude=['number']).columns) > 0 else None
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nGrouped by '{group_field_id}': mean of '{numeric_field_id}':")
            print(grouped_df.head())
        else:
            print("\nNo suitable group_field_id found for grouping.")
    else:
        print("No numeric field found in first record set.")
else:
    print("No dataframes with loaded records available for analysis.")

## 5. Visualization
Visualize numeric distributions and relationships. Update the below to reference valid `@id` fields from your data, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and numeric_field_id is not None:
    plt.figure(figsize=(7, 4))
    sns.histplot(filtered_df[numeric_field_id], kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion
This notebook demonstrated loading and exploring the FAIR² dataset using Croissant with the `mlcroissant` Python library. 

- Metadata and all data entities were referenced via their `@id`, per Croissant best practices.
- DataFrames were loaded dynamically for each record set and exploratory analysis steps were demonstrated, including filtering, normalization, and grouping.
- Replace field `@id`s in EDA and visualization cells for domain-specific insights as you further analyze this dataset in your own research or application.